# H1 ABSA-RTS Pipeline (Colab v2)

This notebook runs the H1 negation/contrast diagnostics end-to-end with a repo sync step.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/My Drive/AutoResearch/AutoResearch-Copilot')
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'PROJECT_ROOT not found: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Working directory:', PROJECT_ROOT)

Mounted at /content/drive
Working directory: /content/drive/My Drive/AutoResearch/AutoResearch-Copilot


## Sync latest code from GitHub

In [2]:
!git --no-pager pull --ff-only

Already up to date.


In [3]:
RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'h1-negation-contrast-diagnostics' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results dir:', RESULTS_DIR)

Results dir: /content/drive/My Drive/AutoResearch/AutoResearch-Copilot/experiments/h1-negation-contrast-diagnostics/results


## 1) Build ABSA-RTS (fixed negation templates)

In [4]:
!PYTHONPATH=src python -m absa_rts.build_rts \
  --input data/absa_rts_seed_restaurants.csv \
  --output-dir experiments/h1-negation-contrast-diagnostics/results \
  --domain restaurants

Generated 19 pairs and 38 evaluation rows in experiments/h1-negation-contrast-diagnostics/results


## 2) Generate A1 predictions (placeholder)

Replace this file later with your real A1 model outputs.

In [5]:
NOISE = 0.10
SEED = 17

!PYTHONPATH=src python -m absa_rts.generate_predictions \
  --eval-csv experiments/h1-negation-contrast-diagnostics/results/absa_rts_eval.csv \
  --output-csv experiments/h1-negation-contrast-diagnostics/results/a1_predictions.csv \
  --noise {NOISE} \
  --seed {SEED}

Wrote 38 predictions to experiments/h1-negation-contrast-diagnostics/results/a1_predictions.csv


## 3) Evaluate A0 vs A1

In [6]:
!PYTHONPATH=src python -m absa_rts.evaluate_baselines \
  --eval-csv experiments/h1-negation-contrast-diagnostics/results/absa_rts_eval.csv \
  --pairs-csv experiments/h1-negation-contrast-diagnostics/results/absa_rts_pairs.csv \
  --a1-predictions experiments/h1-negation-contrast-diagnostics/results/a1_predictions.csv \
  --output-dir experiments/h1-negation-contrast-diagnostics/results

Wrote metrics to experiments/h1-negation-contrast-diagnostics/results


In [7]:
import json
from pathlib import Path

summary_path = Path('experiments/h1-negation-contrast-diagnostics/results/baseline_summary.csv')
metrics_path = Path('experiments/h1-negation-contrast-diagnostics/results/baseline_metrics.json')
print(summary_path.read_text(encoding='utf-8'))
print('\nDetailed metrics:\n')
print(json.dumps(json.loads(metrics_path.read_text(encoding='utf-8')), indent=2))

model,accuracy,macro_f1,metamorphic_pass_rate
A0,1.0000,1.0000,1.0000
A1,0.9211,0.8286,1.0000


Detailed metrics:

{
  "models": [
    {
      "model": "A0",
      "accuracy": 1.0,
      "macro_f1": 1.0,
      "metamorphic_pass_rate": 1.0,
      "category_metrics": {
        "negation": {
          "accuracy": 1.0,
          "macro_f1": 0.6666666666666666,
          "metamorphic_pass_rate": 1.0
        },
        "contrast": {
          "accuracy": 1.0,
          "macro_f1": 1.0,
          "metamorphic_pass_rate": 1.0
        }
      }
    },
    {
      "model": "A1",
      "accuracy": 0.9210526315789473,
      "macro_f1": 0.8285714285714286,
      "metamorphic_pass_rate": 1.0,
      "category_metrics": {
        "negation": {
          "accuracy": 0.9444444444444444,
          "macro_f1": 0.6470588235294118,
          "metamorphic_pass_rate": 1.0
        },
        "contrast": {
          "accuracy": 0.9,
          "macro_f1": 0.8518518518518517,
          "metamorphic_pass_rate": 1.